In [1]:
import os
import re

import GEOparse
import pandas as pd

# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {}
    for item in value.metadata['characteristics_ch1']:
        k, v = item.split(':', 1)
        metadata_object[k.strip()] = v.strip()

    metadata[sample_title] = metadata_object


# Create metadata DataFrame and join as columns to df_genes (rows=samples)
metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'


# Use author provided expression data


In [2]:

import tarfile
import gzip
import urllib.request

# Download the tar archive
url = "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE190113&format=file"
urllib.request.urlretrieve(url, "GSE190113_RAW.tar")

# Extract and load files
def load_geo_data(tar_path):
    dataframes_tpm = {}
    dataframes_combat = {}
    with tarfile.open(tar_path, 'r') as tar:
        for member in tar.getmembers():
            if member.name.endswith('.tsv.gz'):
                # Extract file
                f = tar.extractfile(member)

                # Decompress and read
                with gzip.open(f, 'rt') as gz_file:
                    df = pd.read_csv(gz_file, sep='\t', index_col=0)
                    
                # Clean filename for dict key
                key = re.sub(r'^GSM\d+_', '', member.name.replace('.tsv.gz', ''))

                if key.endswith('.TPM'):
                    dataframes_tpm[key] = df
                elif key.endswith('.ComBatAdjusted'):
                    dataframes_combat[key] = df
    
    return dataframes_tpm, dataframes_combat

# Load all files
tpm_data, combat_data = load_geo_data("GSE190113_RAW.tar")

tpm_data_combined = pd.concat(tpm_data.values(), axis=1)
tpm_data_combined = tpm_data_combined.T
tpm_data_combined.index.name = "Sample"
tpm_data_combined.columns.name = None

combat_data_combined = pd.concat(combat_data.values(), axis=1)
combat_data_combined = combat_data_combined.T
combat_data_combined.index.name = "Sample"
combat_data_combined.columns.name = None


In [3]:
data_file_name = f'{GEO_ID}_original.csv'
# data_file_name_combat = f'{GEO_ID}_COMBAT_ADJUSTED.csv'

metadata_df = metadata_df.reindex(combat_data_combined.index)
missing_metadata = metadata_df.index[metadata_df.isna().all(axis=1)]
if not missing_metadata.empty:
    raise ValueError(f"Missing metadata for expression samples: {missing_metadata.tolist()}")

# Drop metadata columns that only describe the four scRNA-seq samples absent from the bulk expression archive.
bulk_metadata_df = metadata_df.dropna(axis=1, how='all')

# df_tpm = bulk_metadata_df.join(tpm_data_combined, how='inner')
# df_tpm.reset_index(inplace=True)
# df_tpm.to_csv(data_file_name, index=False)

df_combat = bulk_metadata_df.join(combat_data_combined, how='inner')
df_combat.reset_index(inplace=True)
df_combat.to_csv(data_file_name, index=False)
df_combat


,Sample,gender,age at diagnosis (years),mutation category,subtype,primary tumor location,cohort center,site of tumor sampling,ethnicity,ajcc stage,...,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
0,BAIX,M,48,BRAF,SunExposed,right occipital lobe (scalp),Yale,Metastatic,Caucasian,4,...,4.238683,5.644566,6.233697,1.819664,2.430716,5.270749,1.157515,5.129748,6.262725,6.327488
1,BAY,M,63,WT,SunExposed,right lateral arm,Yale,Metastatic,Caucasian,4,...,3.369043,5.044671,5.661581,2.281543,2.968554,4.835667,1.755134,4.864588,4.854396,5.872085
2,BILI,M,39,WT,SunExposed,left ear (tragus),Yale,Metastatic,Caucasian,4,...,3.831359,4.721497,4.685914,2.305708,2.726233,4.244033,0.718545,4.291452,4.525052,5.248131
3,BROT,M,71,WT,SunExposed,right shoulder,Yale,Metastatic,Caucasian,4,...,4.318865,5.368309,5.091451,1.646034,2.601106,3.944951,0.920962,4.061010,7.017117,5.386654
4,CHER,M,55,RAS,SunExposed,right arm,Yale,Metastatic,Caucasian,4,...,3.278025,4.431412,5.114594,2.005626,2.485849,4.602899,0.851397,4.473105,7.566240,5.258804
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,ZIRG,M,33,WT,SunExposed,left arm,Yale,Primary,Caucasian,3,...,2.893567,4.755142,5.139684,1.540106,2.430198,4.382423,0.565540,3.316913,6.770244,4.811157
101,ZS-2,M,NA,NA,Acral,foot,CSU,Primary,Asian,2,...,3.121889,5.266913,6.246904,1.941702,2.409195,4.198242,0.893906,4.308468,6.207457,5.618519
102,ZS-3,M,53,NA,Acral,right foot,CSU,Primary,Asian,4,...,3.953576,5.329582,5.621776,1.966089,2.612857,4.077412,0.790160,4.365297,5.678807,4.882114
103,ZS-4,M,46,NA,Acral,right foot,CSU,Primary,Asian,2,...,4.331990,5.458433,3.063367,2.202520,2.495591,3.685359,0.942240,5.244817,4.620405,5.756973


# Generate description

In [4]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)
    print("Description generated")

desc_data['data_summary'] = {
    'samples': metadata_df.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_combat.shape[1]
}

print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated
